
# 18 — Prompt Engineering for Evidence-Based Claim Verification

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from:**
- Claim review and fact extraction
- Evidence-grounded verification
- Structured model outputs

## Goal

Evaluate how prompt design influences an LLM's
ability to verify factual statements.

Using Gemini, we will:

1. Prepare six factual claims.
2. Provide independently defined evidence records.
3. Ask Gemini to assess each claim.
4. Compare a baseline and an improved prompt.
5. Validate structured JSON responses.
6. Evaluate verdict accuracy and evidence citations.

Possible verdicts:

- `supported`
- `contradicted`
- `insufficient_evidence`

The improved prompt adds explicit verification
rules, evidence requirements and uncertainty
handling.

The experiment uses fictional business information.
It does not access real company data or verify
live news.

The purpose is to test evidence-based reasoning
and prompt instructions, not a web-search system.



## 1. Claim Extraction vs. Claim Verification

A **claim** is a statement that can potentially
be checked against evidence.

Example:

"Revenue increased by 25% in 2025."

A language model can identify this as a factual
claim, but identifying it does not establish
whether it is true.

**Verification** requires relevant evidence.

Three verdicts are possible.

### Supported

The available evidence directly establishes
the claim.

### Contradicted

The available evidence directly conflicts
with the claim.

### Insufficient evidence

The available evidence does not establish
whether the claim is true or false.

This third category is particularly important.

A model should not interpret missing information
as proof that a claim is false.

Similarly, a claim should not be considered
supported merely because it sounds plausible.

### Prompt-engineering question

Does a prompt containing explicit evidence rules,
citation requirements and abstention instructions
perform differently from a basic verification prompt?


In [1]:

import json
import pandas as pd

EVIDENCE = [
    {
        "id": "E1",
        "source": "2025 Annual Financial Statement",
        "text": (
            "HarborCart reported revenue of "
            "€10 million for 2025."
        )
    },
    {
        "id": "E2",
        "source": "2024 Annual Financial Statement",
        "text": (
            "HarborCart reported revenue of "
            "€8 million for 2024."
        )
    },
    {
        "id": "E3",
        "source": "2025 Delivery Report",
        "text": (
            "Average standard delivery time "
            "during 2025 was 4 business days."
        )
    },
    {
        "id": "E4",
        "source": "Returns Policy",
        "text": (
            "The standard return window "
            "is 14 days after purchase."
        )
    },
    {
        "id": "E5",
        "source": "Express Delivery Service Notice",
        "text": (
            "Express delivery is available "
            "in Dublin. This notice does not "
            "specify any expansion plans."
        )
    },
    {
        "id": "E6",
        "source": "2025 Customer Satisfaction Survey",
        "text": (
            "The customer satisfaction "
            "score for 2025 was 82%."
        )
    }
]

evidence_df = pd.DataFrame(EVIDENCE)

print(evidence_df.to_string(index=False))

assert len(EVIDENCE) == 6
assert evidence_df["id"].is_unique


id                            source                                                                                       text
E1   2025 Annual Financial Statement                                       HarborCart reported revenue of €10 million for 2025.
E2   2024 Annual Financial Statement                                        HarborCart reported revenue of €8 million for 2024.
E3              2025 Delivery Report                            Average standard delivery time during 2025 was 4 business days.
E4                    Returns Policy                                      The standard return window is 14 days after purchase.
E5   Express Delivery Service Notice Express delivery is available in Dublin. This notice does not specify any expansion plans.
E6 2025 Customer Satisfaction Survey                                          The customer satisfaction score for 2025 was 82%.


In [2]:

CLAIMS = [
    {
        "id": "C1",
        "claim": (
            "HarborCart's revenue increased "
            "by 25% from 2024 to 2025."
        )
    },
    {
        "id": "C2",
        "claim": (
            "HarborCart allows standard "
            "product returns within 30 days."
        )
    },
    {
        "id": "C3",
        "claim": (
            "Average standard delivery "
            "time in 2025 was 4 business days."
        )
    },
    {
        "id": "C4",
        "claim": (
            "HarborCart will expand express "
            "delivery to Cork in 2026."
        )
    },
    {
        "id": "C5",
        "claim": (
            "HarborCart's customer satisfaction "
            "score in 2025 was 90%."
        )
    },
    {
        "id": "C6",
        "claim": (
            "HarborCart reduced all product "
            "prices by 10% in 2025."
        )
    }
]

REFERENCE = {
    "C1": {
        "verdict": "supported",
        "evidence_ids": {"E1", "E2"}
    },
    "C2": {
        "verdict": "contradicted",
        "evidence_ids": {"E4"}
    },
    "C3": {
        "verdict": "supported",
        "evidence_ids": {"E3"}
    },
    "C4": {
        "verdict": "insufficient_evidence",
        "evidence_ids": set()
    },
    "C5": {
        "verdict": "contradicted",
        "evidence_ids": {"E6"}
    },
    "C6": {
        "verdict": "insufficient_evidence",
        "evidence_ids": set()
    }
}

claims_df = pd.DataFrame(CLAIMS)

print(claims_df.to_string(index=False))

assert len(CLAIMS) == 6
assert set(claims_df["id"]) == set(REFERENCE)


id                                                       claim
C1    HarborCart's revenue increased by 25% from 2024 to 2025.
C2  HarborCart allows standard product returns within 30 days.
C3 Average standard delivery time in 2025 was 4 business days.
C4    HarborCart will expand express delivery to Cork in 2026.
C5   HarborCart's customer satisfaction score in 2025 was 90%.
C6       HarborCart reduced all product prices by 10% in 2025.



## 2. Experiment Design

We will compare two prompts using:

- The same six claims.
- The same six evidence records.
- The same Gemini model.
- The same required response format.

The reference verdicts are defined locally
and are not included in either model prompt.

### Baseline prompt

Provides a short verification instruction
and the allowed verdict labels.

### Improved prompt

Adds:

- Explicit definitions of each verdict.
- Instructions to use only supplied evidence.
- Rules for missing information.
- Requirements for evidence citations.
- Special handling for numerical comparisons.
- A separate worked example.

### Evaluation

We will measure:

1. Verdict accuracy.
2. Exact agreement with expected evidence IDs.
3. Output validation errors.
4. Changes between baseline and improved predictions.

Exact citation agreement is deliberately strict.
A citation must identify the expected evidence
records, not just any relevant-looking source.

The example is small, so results should not
be generalized to broader fact-checking tasks.


In [3]:

%pip install -q openai

from getpass import getpass
from openai import OpenAI

MODEL = "gemini-3.5-flash-lite"

api_key = getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=api_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    max_retries=0,
    timeout=120
)


def verify_with_gemini(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    content = response.choices[0].message.content

    if not content or not content.strip():
        raise ValueError("Gemini returned an empty response.")

    text = content.strip()

    # Remove Markdown code fences if present.
    if text.startswith("```"):
        lines = text.splitlines()

        if lines[-1].strip() == "```":
            text = "\n".join(lines[1:-1]).strip()

    return json.loads(text)


experiment_input = json.dumps(
    {
        "evidence": EVIDENCE,
        "claims": CLAIMS
    },
    indent=2,
    ensure_ascii=False
)

print("Gemini configured:", MODEL)
print("Verification input prepared.")


Enter your Gemini API key: ··········
Gemini configured: gemini-3.5-flash-lite
Verification input prepared.



## 3. Baseline Verification Prompt

The baseline prompt gives Gemini a straightforward
fact-checking task.

It defines the three allowed verdicts and asks
for evidence references.

However, it does not explain how to handle:

- Missing information about future events.
- Claims that require calculations.
- Differences between contradiction and
  insufficient evidence.
- Multiple documents needed for one conclusion.

This provides a simple starting point for
comparing the improved prompt.

The model must return a structured JSON object
containing a verdict, evidence IDs and explanation
for every claim.


In [4]:

baseline_prompt = """
You are a fact-checking assistant.

Verify each claim using the provided evidence.

Use exactly one verdict:
- supported
- contradicted
- insufficient_evidence

For every claim, return its ID, verdict,
supporting evidence IDs and a brief explanation.

Return only valid JSON in this format:

{
  "results": [
    {
      "claim_id": "C1",
      "verdict": "supported",
      "evidence_ids": ["E1"],
      "explanation": "Brief explanation"
    }
  ]
}

Include every claim once.
Do not include Markdown.
"""

if "baseline_result" not in globals():
    baseline_result = verify_with_gemini(
        baseline_prompt,
        experiment_input
    )

    print("Baseline verification completed.")
else:
    print("Using saved baseline results.")

print(json.dumps(baseline_result, indent=2))


Baseline verification completed.
{
  "results": [
    {
      "claim_id": "C1",
      "verdict": "supported",
      "evidence_ids": [
        "E1",
        "E2"
      ],
      "explanation": "Revenue was \u20ac8 million in 2024 and \u20ac10 million in 2025, which represents a 25% increase."
    },
    {
      "claim_id": "C2",
      "verdict": "contradicted",
      "evidence_ids": [
        "E4"
      ],
      "explanation": "The returns policy states that the standard return window is 14 days, not 30 days."
    },
    {
      "claim_id": "C3",
      "verdict": "supported",
      "evidence_ids": [
        "E3"
      ],
      "explanation": "The delivery report explicitly states the average standard delivery time in 2025 was 4 business days."
    },
    {
      "claim_id": "C4",
      "verdict": "contradicted",
      "evidence_ids": [
        "E5"
      ],
      "explanation": "The notice states that express delivery is available in Dublin and does not specify any expansion plans."
    


## 4. Improved Evidence-Grounded Prompt

The improved prompt introduces stricter
verification instructions.

### Evidence-only reasoning

Gemini must use the supplied records rather
than outside knowledge or assumptions.

### Explicit abstention

If the evidence cannot establish a claim,
the model must return `insufficient_evidence`.

### Numerical verification

Claims involving percentage changes require
the relevant source values and a correct calculation.

### Evidence traceability

The model must identify the records needed
to justify its decision.

### Separate worked example

A short example demonstrates the difference
between insufficient evidence and contradiction.

It uses an unrelated company, so it does not
reveal the answers for the evaluation claims.

These rules are designed to improve consistency,
but their effectiveness will be measured rather
than assumed.


In [5]:

improved_prompt = """
You are an evidence-based claim verification analyst.

Your task is to assess factual claims using ONLY
the supplied evidence records.

VERDICTS

supported:
The evidence directly establishes the claim.

contradicted:
The evidence directly establishes that the
claim is false or conflicts with the claim.

insufficient_evidence:
The evidence is missing, incomplete or unable
to establish either support or contradiction.

VERIFICATION RULES

1. Treat evidence records as the only information
   available for this task.
2. Do not use general knowledge or assumptions.
3. Do not treat absence of information as proof
   that a claim is false.
4. Do not infer future plans from current services.
5. For numerical claims, verify values, units,
   periods and calculations.
6. Cite every evidence record necessary to
   justify a supported or contradicted verdict.
7. Do not cite unrelated evidence records.
8. For insufficient_evidence, return an empty
   evidence_ids list. Explain what is missing.
9. Every claim must receive exactly one verdict.
10. Keep explanations concise and factual.

WORKED EXAMPLE

Evidence:
R1: "Northstar's warranty covers 12 months."

Claim:
"Northstar plans to offer a 24-month warranty next year."

Correct result:
{
  "verdict": "insufficient_evidence",
  "evidence_ids": [],
  "explanation": "The current warranty does not
  establish future warranty plans."
}

The example is separate from the task data.

OUTPUT FORMAT

Return valid JSON only:

{
  "results": [
    {
      "claim_id": "C1",
      "verdict": "supported",
      "evidence_ids": ["E1"],
      "explanation": "Brief evidence-based explanation."
    }
  ]
}

The output examples illustrate JSON structure,
not the expected answers for the task.

REQUIREMENTS

- Include every claim ID exactly once.
- Use only the three permitted verdict labels.
- Use only evidence IDs from the supplied records.
- Do not include Markdown or extra commentary.
"""

if "improved_result" not in globals():
    improved_result = verify_with_gemini(
        improved_prompt,
        experiment_input
    )

    print("Improved verification completed.")
else:
    print("Using saved improved results.")

print(json.dumps(improved_result, indent=2))


Improved verification completed.
{
  "results": [
    {
      "claim_id": "C1",
      "verdict": "supported",
      "evidence_ids": [
        "E1",
        "E2"
      ],
      "explanation": "Revenue was \u20ac8 million in 2024 and \u20ac10 million in 2025, which represents a 25% increase."
    },
    {
      "claim_id": "C2",
      "verdict": "contradicted",
      "evidence_ids": [
        "E4"
      ],
      "explanation": "The standard return window is 14 days, not 30 days."
    },
    {
      "claim_id": "C3",
      "verdict": "supported",
      "evidence_ids": [
        "E3"
      ],
      "explanation": "The delivery report states that the average standard delivery time during 2025 was 4 business days."
    },
    {
      "claim_id": "C4",
      "verdict": "insufficient_evidence",
      "evidence_ids": [],
      "explanation": "The evidence mentions express delivery in Dublin but does not contain information about expansion to Cork in 2026."
    },
    {
      "claim_id": "C5",
 


## 5. Evaluate Verdicts and Evidence References

A correct verdict is important, but it is
not enough for reliable claim verification.

For example, Gemini might correctly identify
a revenue-growth claim as supported while
citing only the current year's revenue.

That would not provide enough evidence
to verify the percentage change.

We therefore measure two separate properties:

**Verdict accuracy**

Does the predicted label match the reference?

**Evidence-reference accuracy**

Does the prediction cite exactly the evidence
records needed for the reference decision?

A complete result must satisfy both.

We also validate that the output contains
all claim IDs, recognized verdicts, and
valid evidence references.

Malformed or missing predictions do not
receive credit.


In [6]:

ALLOWED_VERDICTS = {
    "supported",
    "contradicted",
    "insufficient_evidence"
}

VALID_EVIDENCE_IDS = {
    item["id"] for item in EVIDENCE
}


def parse_verification_result(result):
    predictions = {}
    issues = []

    if not isinstance(result, dict):
        return {}, ["Response is not a JSON object"]

    entries = result.get("results")

    if not isinstance(entries, list):
        return {}, ["Missing or invalid results list"]

    seen = set()

    for entry in entries:
        if not isinstance(entry, dict):
            issues.append("Non-object result entry")
            continue

        claim_id = entry.get("claim_id")

        if claim_id not in REFERENCE:
            issues.append(f"Unknown claim ID: {claim_id!r}")
            continue

        if claim_id in seen:
            issues.append(f"Duplicate claim ID: {claim_id}")
            predictions[claim_id] = None
            continue

        seen.add(claim_id)

        verdict = entry.get("verdict")
        evidence_ids = entry.get("evidence_ids")
        explanation = entry.get("explanation")

        if verdict not in ALLOWED_VERDICTS:
            issues.append(f"Invalid verdict for {claim_id}")
            predictions[claim_id] = None
            continue

        if not isinstance(evidence_ids, list):
            issues.append(f"Invalid evidence list for {claim_id}")
            predictions[claim_id] = None
            continue

        if not all(
            isinstance(value, str)
            and value in VALID_EVIDENCE_IDS
            for value in evidence_ids
        ):
            issues.append(f"Unknown evidence ID for {claim_id}")
            predictions[claim_id] = None
            continue

        if len(evidence_ids) != len(set(evidence_ids)):
            issues.append(f"Duplicate evidence ID for {claim_id}")
            predictions[claim_id] = None
            continue

        if not isinstance(explanation, str) or not explanation.strip():
            issues.append(f"Missing explanation for {claim_id}")
            predictions[claim_id] = None
            continue

        predictions[claim_id] = {
            "verdict": verdict,
            "evidence_ids": set(evidence_ids),
            "explanation": explanation
        }

    missing = set(REFERENCE) - set(predictions)

    if missing:
        issues.append(f"Missing claims: {sorted(missing)}")

    return predictions, issues


baseline_predictions, baseline_issues = (
    parse_verification_result(baseline_result)
)

improved_predictions, improved_issues = (
    parse_verification_result(improved_result)
)


def get_verdict(predictions, claim_id):
    prediction = predictions.get(claim_id)
    return prediction["verdict"] if prediction else None


def evaluate_predictions(predictions):
    verdict_correct = 0
    evidence_correct = 0
    fully_correct = 0

    for claim_id, reference in REFERENCE.items():
        prediction = predictions.get(claim_id)

        if prediction is None:
            continue

        verdict_ok = (
            prediction["verdict"] == reference["verdict"]
        )

        evidence_ok = (
            prediction["evidence_ids"]
            == reference["evidence_ids"]
        )

        verdict_correct += int(verdict_ok)
        evidence_correct += int(evidence_ok)
        fully_correct += int(verdict_ok and evidence_ok)

    return {
        "verdicts": verdict_correct,
        "evidence": evidence_correct,
        "complete": fully_correct
    }


baseline_scores = evaluate_predictions(
    baseline_predictions
)

improved_scores = evaluate_predictions(
    improved_predictions
)


rows = []

for claim_id in REFERENCE:
    rows.append({
        "Claim": claim_id,
        "Expected": REFERENCE[claim_id]["verdict"],
        "Baseline": get_verdict(
            baseline_predictions, claim_id
        ),
        "Improved": get_verdict(
            improved_predictions, claim_id
        )
    })

comparison_df = pd.DataFrame(rows)

print(comparison_df.to_string(index=False))

print("\nEVALUATION")
print("Metric                  Baseline   Improved")

for metric, label in [
    ("verdicts", "Correct verdicts"),
    ("evidence", "Correct evidence IDs"),
    ("complete", "Fully correct results")
]:
    print(
        f"{label:<23}"
        f"{baseline_scores[metric]}/6"
        f"        {improved_scores[metric]}/6"
    )

print("\nBaseline validation issues:")
print(baseline_issues or "None")

print("\nImproved validation issues:")
print(improved_issues or "None")

changed = [
    claim_id
    for claim_id in REFERENCE
    if get_verdict(baseline_predictions, claim_id)
    != get_verdict(improved_predictions, claim_id)
]

print("\nVerdicts changed:", changed or "None")


Claim              Expected              Baseline              Improved
   C1             supported             supported             supported
   C2          contradicted          contradicted          contradicted
   C3             supported             supported             supported
   C4 insufficient_evidence          contradicted insufficient_evidence
   C5          contradicted          contradicted          contradicted
   C6 insufficient_evidence insufficient_evidence insufficient_evidence

EVALUATION
Metric                  Baseline   Improved
Correct verdicts       5/6        6/6
Correct evidence IDs   5/6        6/6
Fully correct results  5/6        6/6

Baseline validation issues:
None

Improved validation issues:
None

Verdicts changed: ['C4']


In [7]:

print("DETAILED IMPROVED-PROMPT RESULTS")
print("=" * 55)

for claim in CLAIMS:
    claim_id = claim["id"]
    prediction = improved_predictions.get(claim_id)

    print(f"\n{claim_id}: {claim['claim']}")
    print("Expected:", REFERENCE[claim_id]["verdict"])

    if prediction is None:
        print("Model output: missing or invalid")
        continue

    print("Predicted:", prediction["verdict"])
    print(
        "Cited evidence:",
        sorted(prediction["evidence_ids"])
    )
    print("Explanation:", prediction["explanation"])


print("\nMANUAL REVIEW CHECKLIST")
print("- Does C1 correctly calculate 25% from €8m to €10m?")
print("- Does C2 recognize the 14-day vs 30-day conflict?")
print("- Does C4 avoid inventing Cork expansion plans?")
print("- Does C5 correctly compare 82% against 90%?")
print("- Does C6 acknowledge the missing pricing evidence?")
print("- Are any explanations unsupported by the records?")


DETAILED IMPROVED-PROMPT RESULTS

C1: HarborCart's revenue increased by 25% from 2024 to 2025.
Expected: supported
Predicted: supported
Cited evidence: ['E1', 'E2']
Explanation: Revenue was €8 million in 2024 and €10 million in 2025, which represents a 25% increase.

C2: HarborCart allows standard product returns within 30 days.
Expected: contradicted
Predicted: contradicted
Cited evidence: ['E4']
Explanation: The standard return window is 14 days, not 30 days.

C3: Average standard delivery time in 2025 was 4 business days.
Expected: supported
Predicted: supported
Cited evidence: ['E3']
Explanation: The delivery report states that the average standard delivery time during 2025 was 4 business days.

C4: HarborCart will expand express delivery to Cork in 2026.
Expected: insufficient_evidence
Predicted: insufficient_evidence
Cited evidence: []
Explanation: The evidence mentions express delivery in Dublin but does not contain information about expansion to Cork in 2026.

C5: HarborCart's 


## Findings and Limitations

### What This Notebook Demonstrates

- Identifying factual claims that require verification.
- Using evidence records to assess claims.
- Distinguishing support, contradiction and
  insufficient evidence.
- Designing baseline and improved verification prompts.
- Requiring structured JSON responses.
- Validating verdicts and evidence references.
- Evaluating predictions against reference answers.
- Inspecting model explanations for unsupported reasoning.

### Prompt Engineering Techniques

**Evidence grounding**

The model is instructed to use only the
supplied evidence.

**Explicit verdict definitions**

The improved prompt specifies the conditions
for each possible decision.

**Abstention**

The model must return `insufficient_evidence`
when the available records cannot establish
a claim's truth or falsity.

**Numerical reasoning**

The prompt requires careful treatment of
numbers, dates, units and calculations.

**Citation requirements**

Model decisions must identify the relevant
evidence records.

**Few-shot prompting**

A separate example demonstrates how to
handle uncertainty about future events.

**Prompt evaluation**

Both prompts receive identical claims and evidence,
and their outputs are evaluated against the same
reference answers.

### Relationship to the Udemy Example

The original claim-review example combines
LLM-based claim extraction with external
evidence retrieval.

This notebook focuses on the verification stage.

It uses a controlled evidence collection
instead of live web search.

The difference is intentional: the experiment
evaluates how prompts affect evidence-based
verdicts without also introducing variability
from search results.


### Observed Results

Both verification prompts were evaluated on six
fictional business claims against the same six
evidence records.

| Metric | Baseline | Improved |
|---|---:|---:|
| Correct verdicts | 5/6 | 6/6 |
| Correct evidence references | 5/6 | 6/6 |
| Fully correct results | 5/6 | 6/6 |
| Invalid or missing outputs | 0 | 0 |

**Key finding — Claim C4**

Claim: "HarborCart will expand express delivery
to Cork in 2026."

- Expected: `insufficient_evidence`
- Baseline: `contradicted`
- Improved: `insufficient_evidence`

The available evidence described express delivery
in Dublin but did not establish whether the company
would expand to Cork.

The baseline incorrectly treated missing information
as evidence against the claim.

The improved prompt correctly distinguished
insufficient evidence from contradiction.

**Other findings**

- Both prompts correctly assessed the other
  five claims.
- The revenue-growth calculation was correct:
  (€10m - €8m) / €8m = 25%.
- The improved prompt cited the expected evidence
  records for all six claims.
- The improved explanations were consistent with
  the supplied evidence.
- Both JSON responses passed local validation.

**Interpretation**

The improved prompt increased verdict accuracy
from 5/6 (83.3%) to 6/6 (100%), a gain of
16.7 percentage points.

This provides a concrete example of the value
of explicit uncertainty-handling instructions.

However, the experiment is too small to establish
general reliability or identify which individual
prompt instruction caused the improvement.



### Limitations

- The company and evidence records are fictional.
- Only six claims are evaluated.
- Evidence retrieval is not implemented.
- The evidence is treated as authoritative
  within the experiment, not independently verified.
- Each prompt is executed once.
- The improved prompt provides more detailed
  task instructions, but the experiment cannot
  isolate which instruction caused any difference.
- Model output is validated locally rather
  than constrained by a provider-enforced schema.
- Exact citation matching may penalize alternative
  evidence sets that are also sufficient.
- Automated checks do not establish that
  explanations are logically correct.
- The experiment does not address source
  credibility, conflicting sources or real-time
  information updates.
- Results cannot establish broad fact-checking
  reliability.

### Conclusion

This notebook demonstrates how prompt engineering
can influence evidence-based claim verification.

The model must distinguish between claims that
are supported, contradicted or unverifiable
from the provided evidence.

The improved prompt adds rules for grounding,
uncertainty, numerical comparisons and citations.

Local evaluation tests both the model's verdicts
and the supporting evidence references.

A reliable claim-review workflow must do more
than produce plausible judgments: its decisions
should be traceable to sufficient evidence.

The observed results determine whether the
additional prompt instructions improved performance
on this controlled test.
